# Conversion analysis

Decomposes task success into two factors:

Success = Routing-Rate × Conversion-Rate


- Routing-Rate = pipeline-attributed routed mass (`success + execution_fail`). Successful episodes enter this mass regardless of whether they used the full gold path.  
- Conversion-Rate = success within that pipeline-attributed routed mass.

Both are derived from the success-first error decomposition: successful episodes are counted as such regardless of path, while failed episodes are attributed to the earliest broken stage. Routing here is therefore an attribution construct, not empirical full-gold coverage.


In [7]:
from pathlib import Path
import sys

analysis_dir = Path.cwd() / "analysis"
if str(analysis_dir) not in sys.path:
    sys.path.insert(0, str(analysis_dir))

import pandas as pd

from metrics.errors import classified_rows
from metrics.conditions import OFFICEBENCH_CONFIG, GAIA_CONFIG

In [8]:
def conversion_table(card: str, config=OFFICEBENCH_CONFIG) -> pd.DataFrame:
    """Routing-Rate x Conversion-Rate table from classified error rows."""
    rows = classified_rows(card, config)
    out = []
    for condition in config.system_order:
        c = rows[rows["Condition"] == condition]["Error"].value_counts()
        n = c.sum()
        n_success  = c.get("success",       0)
        n_exec_fail = c.get("execution_fail", 0)
        n_routed   = n_success + n_exec_fail   # success or post-routing failure
        routing    = n_routed / n if n else float("nan")
        conversion = n_success / n_routed if n_routed else float("nan")
        out.append({
            "Condition":        condition,
            "N":                n,
            "Success (%)": round(n_success / n * 100, 1) if n else float("nan"),
            "Routing (%)": round(routing    * 100,    1),
            "Conversion (%)": round(conversion * 100, 1),
        })
    return pd.DataFrame(out).set_index("Condition")

## OfficeBench

In [9]:
print("OfficeBench — rich cards")
display(conversion_table("rich", OFFICEBENCH_CONFIG))

OfficeBench — rich cards


,N,Success (%),Routing (%),Conversion (%)
Condition,,,,
BL-Lower,542,35.1,52.6,66.7
BL-Upper,542,46.3,64.6,71.7
Blueprint,541,50.6,71.2,71.2
Playbook,541,44.7,60.6,73.8
Adaptive System,542,49.4,69.2,71.5


In [10]:
print("OfficeBench — sparse cards")
display(conversion_table("sparse", OFFICEBENCH_CONFIG))

OfficeBench — sparse cards


,N,Success (%),Routing (%),Conversion (%)
Condition,,,,
BL-Lower,542,29.3,44.8,65.4
BL-Upper,542,35.8,54.2,66.0
Blueprint,541,43.8,68.0,64.4
Playbook,542,41.9,56.6,73.9
Adaptive System,542,44.5,62.9,70.7


### NOTE: Memory reduces routing-attributed failures, but downstream success is not guaranteed

## GAIA

In [11]:
print("GAIA — rich cards")
display(conversion_table("rich", GAIA_CONFIG))

GAIA — rich cards


,N,Success (%),Routing (%),Conversion (%)
Condition,,,,
BL-Lower,300,29.3,72.0,40.7
BL-Upper,300,34.0,78.7,43.2
Blueprint,300,30.3,78.0,38.9
Playbook,300,29.7,85.0,34.9
Adaptive System,300,31.3,83.3,37.6


In [12]:
print("GAIA — sparse cards")
display(conversion_table("sparse", GAIA_CONFIG))

GAIA — sparse cards


,N,Success (%),Routing (%),Conversion (%)
Condition,,,,
BL-Lower,300,29.7,68.0,43.6
BL-Upper,300,32.7,73.3,44.5
Blueprint,300,29.0,74.0,39.2
Playbook,300,30.7,81.7,37.6
Adaptive System,300,30.3,81.0,37.4
